In [4]:
import pandas as pd
import numpy as np
import torch

# AirPulse veri seti
df = pd.read_csv(
    "../data/PRSA_Data_Aotizhongxin_20130301-20170228.csv"
)

# Timestamp oluştur
df["timestamp"] = pd.to_datetime(
    df[["year", "month", "day", "hour"]]
)

# Kronolojik sırala
df = df.sort_values("timestamp").reset_index(drop=True)

print("Toplam kayıt:", len(df))
print("Eksik PM2.5:", df["PM2.5"].isna().sum())
print("İlk tarih:", df["timestamp"].min())
print("Son tarih:", df["timestamp"].max())

Toplam kayıt: 35064
Eksik PM2.5: 925
İlk tarih: 2013-03-01 00:00:00
Son tarih: 2017-02-28 23:00:00


In [5]:
def create_sequences(df, sequence_length=24):
    X, y, target_times = [], [], []

    values = df["PM2.5"].to_numpy()
    timestamps = df["timestamp"].to_numpy(dtype="datetime64[ns]")

    for i in range(sequence_length, len(df)):

        window = values[i-sequence_length:i]
        target = values[i]

        window_times = timestamps[i-sequence_length:i+1]

        # Eksik ölçümleri çıkar
        if np.isnan(window).any() or np.isnan(target):
            continue

        # Saatler ardışık olmalı
        if not np.all(np.diff(window_times) == np.timedelta64(1, "h")):
            continue

        X.append(window.reshape(sequence_length, 1))
        y.append(target)
        target_times.append(timestamps[i])

    return (
        np.array(X, dtype=np.float32),
        np.array(y, dtype=np.float32),
        pd.to_datetime(target_times)
    )


# Bütün geçerli pencereleri oluştur
X_seq, y_seq, target_times = create_sequences(df)

# Bağımsız test başlangıcı
test_start = pd.Timestamp("2016-07-24 20:00:00")

# Hedef zamanına göre test ayrımı
test_mask = target_times >= test_start

X_test_seq = X_seq[test_mask]
y_test_seq = y_seq[test_mask]
test_target_times = target_times[test_mask]

# Kontroller
assert X_test_seq.ndim == 3
assert X_test_seq.shape[1] == 24
assert len(X_test_seq) == len(y_test_seq)
assert (test_target_times >= test_start).all()

print("Toplam geçerli pencere:", len(X_seq))
print("Test pencere sayısı:", len(X_test_seq))
print("X_test_seq:", X_test_seq.shape)
print("y_test_seq:", y_test_seq.shape)
print("İlk test hedefi:", test_target_times.min())

Toplam geçerli pencere: 30062
Test pencere sayısı: 4619
X_test_seq: (4619, 24, 1)
y_test_seq: (4619,)
İlk test hedefi: 2016-07-24 20:00:00


In [6]:
from sklearn.preprocessing import StandardScaler

# Önceki çalışmalarımızdaki aynı sınırlar
val_start = pd.Timestamp("2015-12-18 16:00:00")

# Kronolojik maskeler
train_mask = target_times < val_start

val_mask = (
    (target_times >= val_start) &
    (target_times < test_start)
)

# Training
X_train_seq = X_seq[train_mask]
y_train_seq = y_seq[train_mask]

# Validation
X_val_seq = X_seq[val_mask]
y_val_seq = y_seq[val_mask]

# Scaler yalnızca training üzerinde fit edilir
input_scaler = StandardScaler()
target_scaler = StandardScaler()

input_scaler.fit(X_train_seq.reshape(-1, 1))
target_scaler.fit(y_train_seq.reshape(-1, 1))

# Test verisini aynı scaler ile dönüştür
X_test_scaled = input_scaler.transform(
    X_test_seq.reshape(-1, 1)
).reshape(X_test_seq.shape)

y_test_scaled = target_scaler.transform(
    y_test_seq.reshape(-1, 1)
)

# Sonuçlar
print("Training:", X_train_seq.shape)
print("Validation:", X_val_seq.shape)
print("Test:", X_test_seq.shape)

print("\nÖlçeklendirilmiş test:")
print("X_test_scaled:", X_test_scaled.shape)
print("y_test_scaled:", y_test_scaled.shape)

print("\nScaler istatistikleri:")
print("Input mean:", input_scaler.mean_[0])
print("Input std:", input_scaler.scale_[0])

Training: (21177, 24, 1)
Validation: (4266, 24, 1)
Test: (4619, 24, 1)

Ölçeklendirilmiş test:
X_test_scaled: (4619, 24, 1)
y_test_scaled: (4619, 1)

Scaler istatistikleri:
Input mean: 83.00201633903434
Input std: 76.02307113339165


In [7]:
import torch
import torch.nn as nn

# Tekrarlanabilirlik
torch.manual_seed(42)


# RNN mimarisi
class AirPulseRNN(nn.Module):
    def __init__(self, input_size=1, hidden_size=32):
        super().__init__()

        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        output, hidden = self.rnn(x)

        last_output = output[:, -1, :]

        return self.fc(last_output)


# LSTM mimarisi
class AirPulseLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=32):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        output, (hidden, cell) = self.lstm(x)

        last_hidden = hidden[-1]

        return self.fc(last_hidden)


# Modelleri oluştur
rnn_model = AirPulseRNN()
lstm_model = AirPulseLSTM()

# Kontrol
print("RNN:", type(rnn_model).__name__)
print("LSTM:", type(lstm_model).__name__)

print("\nRNN parametre sayısı:",
      sum(p.numel() for p in rnn_model.parameters()))

print("LSTM parametre sayısı:",
      sum(p.numel() for p in lstm_model.parameters()))

RNN: AirPulseRNN
LSTM: AirPulseLSTM

RNN parametre sayısı: 1153
LSTM parametre sayısı: 4513


In [8]:
from torch.utils.data import TensorDataset, DataLoader

# 1. Training verisini ölçeklendir
X_train_scaled = input_scaler.transform(
    X_train_seq.reshape(-1, 1)
).reshape(X_train_seq.shape)

y_train_scaled = target_scaler.transform(
    y_train_seq.reshape(-1, 1)
)

# 2. Validation verisini ölçeklendir
X_val_scaled = input_scaler.transform(
    X_val_seq.reshape(-1, 1)
).reshape(X_val_seq.shape)

y_val_scaled = target_scaler.transform(
    y_val_seq.reshape(-1, 1)
)

# 3. TensorDataset oluştur
train_dataset = TensorDataset(
    torch.tensor(X_train_scaled, dtype=torch.float32),
    torch.tensor(y_train_scaled, dtype=torch.float32)
)

val_dataset = TensorDataset(
    torch.tensor(X_val_scaled, dtype=torch.float32),
    torch.tensor(y_val_scaled, dtype=torch.float32)
)

# 4. DataLoader oluştur
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

# 5. Kontroller
print("Training örnekleri:", len(train_dataset))
print("Validation örnekleri:", len(val_dataset))

print("\nTraining X:", train_dataset.tensors[0].shape)
print("Training y:", train_dataset.tensors[1].shape)

print("\nValidation X:", val_dataset.tensors[0].shape)
print("Validation y:", val_dataset.tensors[1].shape)

Training örnekleri: 21177
Validation örnekleri: 4266

Training X: torch.Size([21177, 24, 1])
Training y: torch.Size([21177, 1])

Validation X: torch.Size([4266, 24, 1])
Validation y: torch.Size([4266, 1])


In [9]:
from pathlib import Path
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Tekrarlanabilirlik
torch.manual_seed(42)

# RNN modelini yeniden başlat
rnn_model = AirPulseRNN()

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    rnn_model.parameters(),
    lr=0.001
)

epochs = 10

# RNN eğitimi
for epoch in range(epochs):

    rnn_model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        predictions = rnn_model(X_batch)

        loss = criterion(predictions, y_batch)

        loss.backward()

        optimizer.step()

        total_loss += loss.item() * len(X_batch)

    avg_loss = total_loss / len(train_dataset)

    print(f"Epoch {epoch+1}/{epochs} - Train Loss: {avg_loss:.4f}")


# Validation değerlendirmesi
rnn_model.eval()

val_predictions_scaled = []

with torch.no_grad():
    for X_batch, y_batch in val_loader:
        predictions = rnn_model(X_batch)
        val_predictions_scaled.append(predictions.numpy())

val_predictions_scaled = np.concatenate(val_predictions_scaled)

# Orijinal PM2.5 birimine dönüştür
rnn_val_predictions = target_scaler.inverse_transform(
    val_predictions_scaled
).ravel()

# Validation metrikleri
rnn_val_mae = mean_absolute_error(
    y_val_seq,
    rnn_val_predictions
)

rnn_val_rmse = np.sqrt(
    mean_squared_error(y_val_seq, rnn_val_predictions)
)

print("\n=== RNN VALIDATION ===")
print(f"MAE: {rnn_val_mae:.4f}")
print(f"RMSE: {rnn_val_rmse:.4f}")


# Eğitilmiş model ağırlıklarını kaydet
models_dir = Path("../models")
models_dir.mkdir(exist_ok=True)

torch.save(
    rnn_model.state_dict(),
    models_dir / "rnn_state_dict.pth"
)

print("\n✅ RNN modeli kaydedildi.")

Epoch 1/10 - Train Loss: 0.1534
Epoch 2/10 - Train Loss: 0.0635
Epoch 3/10 - Train Loss: 0.0613
Epoch 4/10 - Train Loss: 0.0612
Epoch 5/10 - Train Loss: 0.0611
Epoch 6/10 - Train Loss: 0.0608
Epoch 7/10 - Train Loss: 0.0611
Epoch 8/10 - Train Loss: 0.0611
Epoch 9/10 - Train Loss: 0.0607
Epoch 10/10 - Train Loss: 0.0606

=== RNN VALIDATION ===
MAE: 10.5165
RMSE: 21.1358

✅ RNN modeli kaydedildi.


In [10]:
# Tekrarlanabilirlik
torch.manual_seed(42)

# LSTM modelini yeniden başlat
lstm_model = AirPulseLSTM()

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    lstm_model.parameters(),
    lr=0.001
)

epochs = 10

# LSTM eğitimi
for epoch in range(epochs):

    lstm_model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        predictions = lstm_model(X_batch)

        loss = criterion(predictions, y_batch)

        loss.backward()

        optimizer.step()

        total_loss += loss.item() * len(X_batch)

    avg_loss = total_loss / len(train_dataset)

    print(f"Epoch {epoch+1}/{epochs} - Train Loss: {avg_loss:.4f}")


# Validation değerlendirmesi
lstm_model.eval()

lstm_val_predictions_scaled = []

with torch.no_grad():
    for X_batch, y_batch in val_loader:

        predictions = lstm_model(X_batch)

        lstm_val_predictions_scaled.append(
            predictions.cpu().numpy()
        )

lstm_val_predictions_scaled = np.concatenate(
    lstm_val_predictions_scaled
)

# Orijinal PM2.5 birimine dönüştür
lstm_val_predictions = target_scaler.inverse_transform(
    lstm_val_predictions_scaled
).ravel()

# Validation metrikleri
lstm_val_mae = mean_absolute_error(
    y_val_seq,
    lstm_val_predictions
)

lstm_val_rmse = np.sqrt(
    mean_squared_error(y_val_seq, lstm_val_predictions)
)

print("\n=== LSTM VALIDATION ===")
print(f"MAE: {lstm_val_mae:.4f}")
print(f"RMSE: {lstm_val_rmse:.4f}")


# Eğitilmiş LSTM ağırlıklarını kaydet
torch.save(
    lstm_model.state_dict(),
    models_dir / "lstm_state_dict.pth"
)

print("\n✅ LSTM modeli kaydedildi.")

Epoch 1/10 - Train Loss: 0.1767
Epoch 2/10 - Train Loss: 0.0633
Epoch 3/10 - Train Loss: 0.0601
Epoch 4/10 - Train Loss: 0.0596
Epoch 5/10 - Train Loss: 0.0595
Epoch 6/10 - Train Loss: 0.0592
Epoch 7/10 - Train Loss: 0.0589
Epoch 8/10 - Train Loss: 0.0589
Epoch 9/10 - Train Loss: 0.0587
Epoch 10/10 - Train Loss: 0.0584

=== LSTM VALIDATION ===
MAE: 10.3180
RMSE: 20.5135

✅ LSTM modeli kaydedildi.


In [11]:
import joblib
from pathlib import Path

# 1. Scaler dosyalarını kaydet
joblib.dump(input_scaler, models_dir / "input_scaler.joblib")
joblib.dump(target_scaler, models_dir / "target_scaler.joblib")

# 2. RNN modelini dosyadan yükle
loaded_rnn = AirPulseRNN()

loaded_rnn.load_state_dict(
    torch.load(
        models_dir / "rnn_state_dict.pth",
        map_location="cpu",
        weights_only=True
    )
)

loaded_rnn.eval()

# 3. LSTM modelini dosyadan yükle
loaded_lstm = AirPulseLSTM()

loaded_lstm.load_state_dict(
    torch.load(
        models_dir / "lstm_state_dict.pth",
        map_location="cpu",
        weights_only=True
    )
)

loaded_lstm.eval()

# 4. Kaydedilen scaler dosyalarını yükle
loaded_input_scaler = joblib.load(
    models_dir / "input_scaler.joblib"
)

loaded_target_scaler = joblib.load(
    models_dir / "target_scaler.joblib"
)

# 5. Aynı validation örnekleriyle tahminleri karşılaştır
X_batch, _ = next(iter(val_loader))

rnn_model.eval()
lstm_model.eval()

with torch.no_grad():
    rnn_original = rnn_model(X_batch)
    rnn_reloaded = loaded_rnn(X_batch)

    lstm_original = lstm_model(X_batch)
    lstm_reloaded = loaded_lstm(X_batch)

print("=== REPRODUCIBILITY CHECK ===")

print(
    "RNN aynı tahminleri üretiyor:",
    torch.allclose(rnn_original, rnn_reloaded)
)

print(
    "LSTM aynı tahminleri üretiyor:",
    torch.allclose(lstm_original, lstm_reloaded)
)

print(
    "Input scaler aynı:",
    np.allclose(input_scaler.mean_, loaded_input_scaler.mean_)
)

print(
    "Target scaler aynı:",
    np.allclose(target_scaler.mean_, loaded_target_scaler.mean_)
)

=== REPRODUCIBILITY CHECK ===
RNN aynı tahminleri üretiyor: True
LSTM aynı tahminleri üretiyor: True
Input scaler aynı: True
Target scaler aynı: True


In [12]:
from torch.utils.data import DataLoader, TensorDataset

# Test verisini kaydedilmiş scaler ile dönüştür
X_test_scaled = loaded_input_scaler.transform(
    X_test_seq.reshape(-1, 1)
).reshape(X_test_seq.shape)

# Test DataLoader
test_dataset = TensorDataset(
    torch.tensor(X_test_scaled, dtype=torch.float32)
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

# MODEL A — Naive Forecast
naive_test_predictions = X_test_seq[:, -1, 0].copy()


# MODEL B — RNN
loaded_rnn.eval()

rnn_test_scaled = []

with torch.no_grad():
    for (X_batch,) in test_loader:
        predictions = loaded_rnn(X_batch)
        rnn_test_scaled.append(predictions.cpu().numpy())

rnn_test_scaled = np.concatenate(rnn_test_scaled)

rnn_test_predictions = loaded_target_scaler.inverse_transform(
    rnn_test_scaled
).ravel()


# MODEL C — LSTM
loaded_lstm.eval()

lstm_test_scaled = []

with torch.no_grad():
    for (X_batch,) in test_loader:
        predictions = loaded_lstm(X_batch)
        lstm_test_scaled.append(predictions.cpu().numpy())

lstm_test_scaled = np.concatenate(lstm_test_scaled)

lstm_test_predictions = loaded_target_scaler.inverse_transform(
    lstm_test_scaled
).ravel()


# KONTROLLER
assert len(naive_test_predictions) == len(y_test_seq)
assert len(rnn_test_predictions) == len(y_test_seq)
assert len(lstm_test_predictions) == len(y_test_seq)

print("=== FINAL TEST TAHMİNLERİ ===")

print("Gerçek değerler:", y_test_seq.shape)
print("Naive tahminleri:", naive_test_predictions.shape)
print("RNN tahminleri:", rnn_test_predictions.shape)
print("LSTM tahminleri:", lstm_test_predictions.shape)

print("\nİlk test örneği:")
print("Tarih:", test_target_times[0])
print("Gerçek PM2.5:", y_test_seq[0])
print("Naive:", naive_test_predictions[0])
print("RNN:", rnn_test_predictions[0])
print("LSTM:", lstm_test_predictions[0])

=== FINAL TEST TAHMİNLERİ ===
Gerçek değerler: (4619,)
Naive tahminleri: (4619,)
RNN tahminleri: (4619,)
LSTM tahminleri: (4619,)

İlk test örneği:
Tarih: 2016-07-24 20:00:00
Gerçek PM2.5: 66.0
Naive: 129.0
RNN: 129.02805
LSTM: 129.00732


In [13]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import pandas as pd
import numpy as np

# Metrik hesaplama fonksiyonu
def evaluate_model(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    return mae, rmse


# Naive Forecast
naive_test_mae, naive_test_rmse = evaluate_model(
    y_test_seq,
    naive_test_predictions
)

# RNN
rnn_test_mae, rnn_test_rmse = evaluate_model(
    y_test_seq,
    rnn_test_predictions
)

# LSTM
lstm_test_mae, lstm_test_rmse = evaluate_model(
    y_test_seq,
    lstm_test_predictions
)


# Önceki validation değerlerini de aynı şekilde hesapla
naive_val_mae, naive_val_rmse = evaluate_model(
    y_val_seq,
    X_val_seq[:, -1, 0]
)


# Final karşılaştırma tablosu
final_results = pd.DataFrame({
    "model": ["Naive", "RNN", "LSTM"],
    "validation_samples": [len(y_val_seq)] * 3,
    "test_samples": [len(y_test_seq)] * 3,
    "validation_mae": [
        naive_val_mae,
        rnn_val_mae,
        lstm_val_mae
    ],
    "validation_rmse": [
        naive_val_rmse,
        rnn_val_rmse,
        lstm_val_rmse
    ],
    "test_mae": [
        naive_test_mae,
        rnn_test_mae,
        lstm_test_mae
    ],
    "test_rmse": [
        naive_test_rmse,
        rnn_test_rmse,
        lstm_test_rmse
    ]
})

print("=== AIRPULSE FINAL TEST RESULTS ===")

display(final_results.round(4))

=== AIRPULSE FINAL TEST RESULTS ===


,model,validation_samples,test_samples,validation_mae,validation_rmse,test_mae,test_rmse
0,Naive,4266,4619,10.3748,21.5890,11.1121,21.2815
1,RNN,4266,4619,10.5165,21.1358,11.2278,20.7872
2,LSTM,4266,4619,10.3180,20.5135,11.0416,20.1592


In [14]:
from pathlib import Path

# CSV dosyasının kaydedileceği klasör
reports_dir = Path("../reports")
reports_dir.mkdir(parents=True, exist_ok=True)

csv_path = reports_dir / "final_test_results.csv"

# Sonuçları kaydet
final_results.to_csv(
    csv_path,
    index=False
)

# Kaydedilen dosyayı tekrar oku
saved_results = pd.read_csv(csv_path)

print("✅ Final test sonuçları kaydedildi.")
print("Dosya:", csv_path)

display(saved_results.round(4))

✅ Final test sonuçları kaydedildi.
Dosya: ../reports/final_test_results.csv


,model,validation_samples,test_samples,validation_mae,validation_rmse,test_mae,test_rmse
0,Naive,4266,4619,10.3748,21.5890,11.1121,21.2815
1,RNN,4266,4619,10.5165,21.1358,11.2278,20.7872
2,LSTM,4266,4619,10.3180,20.5135,11.0416,20.1592


In [ ]:
import numpy as np
import pandas as pd
import torch

valid_values = X_test_seq[0, :, 0].copy()

valid_times = pd.date_range(
    end=test_target_times[0] - pd.Timedelta(hours=1),
    periods=24,
    freq="h"
)


def predict_without_validation(timestamps, values):

    values = np.asarray(values, dtype=np.float32)

    scaled = loaded_input_scaler.transform(
        values.reshape(-1, 1)
    )

    X = torch.tensor(
        scaled.reshape(1, -1, 1),
        dtype=torch.float32
    )

    loaded_lstm.eval()

    with torch.no_grad():
        prediction = loaded_lstm(X).cpu().numpy()

    result = loaded_target_scaler.inverse_transform(
        prediction
    )

    return float(result[0, 0])


# HATA 1: 23 saatlik veri
short_values = valid_values[:23]
short_times = valid_times[:23]

# HATA 2: NaN
nan_values = valid_values.copy()
nan_values[5] = np.nan

# HATA 3: Saatler arasında boşluk
gap_times = list(valid_times)

# 12. saatten sonrasını bir saat ileri kaydır
gap_times[12:] = [
    t + pd.Timedelta(hours=1)
    for t in gap_times[12:]
]


# Mevcut davranışları ölç
test_cases = [
    ("23 saatlik veri", short_times, short_values),
    ("NaN içeren veri", valid_times, nan_values),
    ("Zaman boşluğu", gap_times, valid_values)
]

print("=== AIRPULSE — MEVCUT DAVRANIŞ ===")

for name, times, values in test_cases:

    try:
        result = predict_without_validation(times, values)

        print(f"\n{name}")
        print("Durum: Tahmin üretildi")
        print("Sonuç:", result)

    except Exception as e:

        print(f"\n{name}")
        print("Durum: Hata oluştu")
        print("Hata:", type(e).__name__, str(e))

=== AIRPULSE — MEVCUT DAVRANIŞ ===

23 saatlik veri
Durum: Tahmin üretildi
Sonuç: 121.18209075927734

NaN içeren veri
Durum: Tahmin üretildi
Sonuç: nan

Zaman boşluğu
Durum: Tahmin üretildi
Sonuç: 129.00732421875


In [ ]:
from pathlib import Path
import importlib.util


validation_path = Path("../src/validation.py").resolve()

print("Dosya:", validation_path)
print("Dosya mevcut mu?", validation_path.is_file())


spec = importlib.util.spec_from_file_location(
    "airpulse_validation",
    validation_path
)

validation = importlib.util.module_from_spec(spec)
spec.loader.exec_module(validation)


result = validation.validate_pm25_window(
    timestamps=valid_times,
    values=valid_values
)

print("Girdi geçerli:", result)

Dosya: /home/pasali/Desktop/Data/AI-Engineer/Hafta-3/AirPulse/src/validation.py
Dosya mevcut mu? True
Girdi geçerli: True


In [27]:
# Yeni güvenli tahmin fonksiyonu
def predict_with_validation(timestamps, values):

    # Önce girdiyi doğrula
    validation.validate_pm25_window(
        timestamps=timestamps,
        values=values
    )

    # Girdi geçerliyse tahmin üret
    return predict_without_validation(
        timestamps,
        values
    )


# Önceki ve sonraki davranışları karşılaştır
before_rejected = 0
after_rejected = 0

print("=== INPUT VALIDATION IMPROVEMENT ===")

for name, times, values in test_cases:

    # ÖNCEKİ DAVRANIŞ
    try:
        predict_without_validation(times, values)
        before = "Kabul edildi"
    except Exception:
        before = "Reddedildi"
        before_rejected += 1

    # YENİ DAVRANIŞ
    try:
        predict_with_validation(times, values)
        after = "Kabul edildi"
    except ValueError:
        after = "Reddedildi"
        after_rejected += 1

    print(f"\n{name}")
    print("Önce:", before)
    print("Sonra:", after)


print("\n=== SONUÇ ===")
print(f"Önce reddedilen: {before_rejected}/3")
print(f"Sonra reddedilen: {after_rejected}/3")

=== INPUT VALIDATION IMPROVEMENT ===

23 saatlik veri
Önce: Kabul edildi
Sonra: Reddedildi

NaN içeren veri
Önce: Kabul edildi
Sonra: Reddedildi

Zaman boşluğu
Önce: Kabul edildi
Sonra: Reddedildi

=== SONUÇ ===
Önce reddedilen: 0/3
Sonra reddedilen: 3/3
